# FedSwarm — DAY 2: A1, the go/no-go on the framing (Kaggle GPU)

**Before running, attach two inputs** (right sidebar → **+ Add Input**):

1. **Datasets** → `masoudnickparvar/brain-tumor-mri-dataset`
2. **Your Work** → the **Day 1** notebook's output. Required: it holds the gate's results, from
   which this notebook re-derives the fitness fix. Without it, the guard halts the notebook
   rather than run A1 on the broken default.

Then **Accelerator → GPU T4 x2**.

**A1** replaces the colony with random search, coordinate-grid search, PSO and a GA at an
**identical evaluation budget and identical fitness**. 80 cells, ~17 GPU-h — **two sessions**.
Session 2: attach this notebook's own session-1 output too; finished cells are skipped. The 5
cells A1 shares with Day 1's A2 are skipped as well.

**What to expect, written down before it runs:** three local screens found ACO *losing* to every
control (ACO +0.0013 → +0.0066 against random +0.0304, PSO +0.0484, coordinate grid +0.0501; 0 of
3 seeds). The mechanism is known: the desirability signal spans ~0.04 against a level spacing of
0.25, so the greedy branch reproduces the FedAvg point ~70% of the time.

- **ACO loses or ties** → framing A stands as written in `paper/01_INTRODUCTION.md`.
- **ACO significantly beats all four** → delete §1.3 there and paste the alternative framing from
  the end of the same file.


## 1. Setup — clone from GitHub


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/kaggle/working/ResearchPaper"

# BRANCH is not optional. A bare `git clone` takes the DEFAULT branch, `main`, and every config
# and script these notebooks run exists only on the feature branch.
BRANCH = "claude/happy-hamilton-c5jjil"

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

on = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                    capture_output=True, text=True).stdout.strip()
print("branch:", on)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)
if on != BRANCH:
    raise RuntimeError(f"Checked out {on!r}, not {BRANCH!r}.")


In [ ]:
%cd /kaggle/working/ResearchPaper

# flwr[simulation] pulls in ray; installed first and on its own. fedswarm is --no-deps because
# its own pins target the dev machine and have no Kaggle CUDA build -- Kaggle's base image
# already has a newer working torch/numpy. pip still enforces fedswarm's requires-python
# (>=3.11), so an older runtime fails HERE, loudly, rather than hours later.
#
# EXPECT A RED BLOCK here reading "ERROR: pip's dependency resolver does not currently take into
# account all the packages that are installed", listing bigframes, google-colab, gradio,
# grpcio-tools and others. It is HARMLESS: those are Kaggle's own preinstalled packages
# disagreeing with each other about protobuf/rich/starlette versions, none of which this
# project imports. The line after the installs is the one that matters.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"
!pip install -q --no-deps -e .
!pip install -q omegaconf rich
!python -c "import fedswarm, flwr, sys; print('INSTALL OK -- fedswarm importable, flwr', flwr.__version__, '| Python', sys.version.split()[0])"


In [ ]:
import glob
import os
import sys
from pathlib import Path

inputs = sorted(glob.glob("/kaggle/input/*"))
print("inputs mounted:", [Path(p).name for p in inputs] or "NONE")
if not inputs:
    raise RuntimeError(
        "NO DATASET ATTACHED -- this is the one manual step, and nothing above is wrong.\n"
        "  1. In the notebook editor's RIGHT sidebar, find the 'Input' section.\n"
        "  2. Click '+ Add Input'.\n"
        "  3. Search: brain tumor mri dataset   (owner: masoudnickparvar)\n"
        "  4. Click the (+) next to it. It mounts under /kaggle/input/.\n"
        "  5. Run this cell again -- cells 1-2 do not need re-running."
    )

# Pick the input that actually holds the images rather than trusting glob order: once you
# attach a previous session's output, inputs[0] may be a results folder.
DATA_ROOT = next((p for p in inputs if any(Path(p).rglob("Training"))), None)
if DATA_ROOT is None:
    raise RuntimeError(
        "None of the mounted inputs contains a Training/ directory, so none is the MRI "
        f"dataset. Mounted: {[Path(p).name for p in inputs]}."
    )
print("dataset root:", DATA_ROOT)

# `flwr run` executes an INSTALLED COPY of the app, whose __file__ is not this clone, so the
# app resolves data and cache paths against FEDSWARM_REPO_ROOT rather than its own location.
os.environ["FEDSWARM_DATA_ROOT"] = DATA_ROOT
os.environ["FEDSWARM_REPO_ROOT"] = "/kaggle/working/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

sys.path.insert(0, "src")
import torch  # noqa: E402

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible; these sweeps will not finish on CPU. Fix: Notebook options -> "
        "Accelerator -> GPU T4 x2, then Run -> Restart & clear cell outputs."
    )
print("CPU cores:", os.cpu_count(), "| Python:", sys.version.split()[0])

import flwr  # noqa: E402
from fedswarm.data.download import find_split_parent  # noqa: E402

print("flwr:", flwr.__version__, "| torch:", torch.__version__)
SPLIT_PARENT = find_split_parent(Path(DATA_ROOT))
print("split parent:", SPLIT_PARENT)

# Verify every image is BYTE-IDENTICAL to the one the committed split was built from. A missing
# file would fail loudly later; a re-versioned Kaggle dataset with the same filenames and
# different images would not -- every pseudo-patient boundary would then describe images you
# are not training on, and no result file would show it. ~165 MB of reads, a few seconds.
import hashlib  # noqa: E402

import pandas as pd  # noqa: E402

_manifest = pd.read_csv("data/processed/manifest.csv", dtype={"sha256": str})
missing, changed = [], []
for rel, digest in zip(_manifest["path"], _manifest["sha256"]):
    f = SPLIT_PARENT / rel
    if not f.exists():
        missing.append(rel)
    elif hashlib.sha256(f.read_bytes()).hexdigest() != digest:
        changed.append(rel)
ok = len(_manifest) - len(missing) - len(changed)
print(f"dataset check: {ok}/{len(_manifest)} images present and byte-identical to the manifest")
if missing or changed:
    raise RuntimeError(
        f"{len(missing)} manifest image(s) missing and {len(changed)} with different content "
        f"(first few: {(missing + changed)[:3]}). The attached dataset is not the version the "
        "split was built from -- check you added masoudnickparvar/brain-tumor-mri-dataset and "
        "not a fork, and that Kaggle has not published a new version of it."
    )


### Restore — brings back Day 1's gate results (required) and any earlier A1 session


In [ ]:
# Self-contained on purpose: a Run-All from the middle must not NameError here.
import glob
import shutil
from pathlib import Path

RESULTS = Path("/kaggle/working/ResearchPaper/results")
RESULTS.mkdir(parents=True, exist_ok=True)

restored = 0
for prior in glob.glob("/kaggle/input/**/results", recursive=True):
    for src in Path(prior).rglob("*.json*"):
        dst = RESULTS / src.relative_to(prior)
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            restored += 1
print(f"restored {restored} file(s) from attached outputs")
for sub in ("gate", "ablation", "main", "robustness"):
    d = RESULTS / "fl" / sub
    print(f"  results/fl/{sub}: {len(list(d.glob('*.json'))) if d.exists() else 0}")


## 2. Build the image cache (~3 min, once per session)


In [ ]:
import pandas as pd

from fedswarm.data.cache import build_and_save_cache
from fedswarm.data.download import find_split_parent, resolve_root

manifest = pd.read_csv("data/processed/manifest.csv")
print("manifest rows:", len(manifest), "| pseudo-patients:", manifest.pseudo_patient_id.nunique())
cache_path = build_and_save_cache(manifest, find_split_parent(resolve_root(None)), 112)
print("cache:", cache_path)


## Federation, GPU, and the gate-fix guard — **not optional**

**GPU.** Ray hides the GPU from any actor requested with `num_gpus=0`, so with no fraction set
every client trains on **CPU** while the server keeps the card — and every logged metric looks
normal. `1/num_clients` lets all ten clients share one T4.

**The gate fix.** Day 1's gate chooses a fitness fix and writes it into `pyproject.toml` —
**inside that Kaggle session only**. It is never pushed to GitHub, so every fresh clone,
including a restart of this notebook, starts on the *broken* default. `ensure_gate_fix()`
re-derives the same patch from the gate's result files (restored from Day 1's saved output) and
re-applies it. It is idempotent, so every cell that runs or reads a FedACO sweep calls it first.

It **raises** instead of printing, because a failing `!` line does not stop Run-All — the
notebook would carry straight on into hours of GPU on the broken objective.


In [ ]:
import subprocess
import sys

GPUS_PER_CLIENT = 0.1   # 1/10 clients


def ensure_gate_fix():
    """Apply the gate's fitness fix to pyproject.toml, or halt Run-All."""
    r = subprocess.run(
        [sys.executable, "scripts/apply_gate_fix.py",
         "--from-results", "results/fl/gate", "--apply-verdict"],
        capture_output=True, text=True,
    )
    print(r.stdout[-3000:])
    if r.stderr.strip():
        print(r.stderr[-1500:])
    if r.returncode != 0:
        raise RuntimeError(
            "No usable fitness fix, so no FedACO sweep may run. Either the gate's results are "
            "not here (attach the Day 1 notebook's output: + Add Input -> Your Work), or the "
            "gate found NO arm that closes the corner -- then re-run the gate with a larger "
            "aco-gamma-entropy before anything else."
        )


print("GPUs per ClientApp:", GPUS_PER_CLIENT)


---
# STEP 1 — re-apply Day 1's fitness fix

Halts if Day 1's output is not attached, or if Day 1's gate found no fix.


In [ ]:
ensure_gate_fix()


---
# STEP 2 — A1: 80 cells, ~17 GPU-h


In [ ]:
ensure_gate_fix()
!python scripts/run_sweep_granular.py --config configs/experiment/ablation_a1_reduced.yaml --gpus-per-client {GPUS_PER_CLIENT}


---
# STEP 3 — did ACO beat the controls?

Paired by seed: the same seed gives every method the same partition and initialisation, so
"ACO beat PSO on seed 3" is a like-for-like comparison. Significance is `make_tables`' job (next
cell); this is the direct reading.


In [ ]:
# Filter by the run_ids THIS sweep's runner would produce, via the same code path. Other
# sweeps write to the same directory -- A1 and A2 both use results/fl/ablation -- so filtering
# on config values would fold one into the other. ensure_gate_fix() runs before the ids are
# computed, because run_ids hash pyproject's defaults and the fix changes them.
import json
from collections import defaultdict
from pathlib import Path

from fedswarm.sweep import granular_runs, planned_run_ids

ensure_gate_fix()

CONFIG = "configs/experiment/ablation_a1_reduced.yaml"
mine = planned_run_ids(granular_runs(CONFIG, Path(".").resolve()), "pyproject.toml")
print(f"{len(mine)} cells belong to {Path(CONFIG).name}")

f1 = {}   # label -> final test macro-F1
for path in Path("results/fl").rglob("*.json"):
    result = json.loads(path.read_text())
    if result.get("run_id") in mine:
        value = (result.get("final") or {}).get("final_test_macro_f1")
        if value is not None:
            f1[mine[result["run_id"]]] = float(value)
print(f"{len(f1)}/{len(mine)} of this sweep's cells have a result\n")


def mean_std(vals):
    m = sum(vals) / len(vals)
    s = (sum((v - m) ** 2 for v in vals) / (len(vals) - 1)) ** 0.5 if len(vals) > 1 else 0.0
    return m, s


scores = defaultdict(dict)   # (partition, seed) -> {method: f1}
for label, value in f1.items():
    method, partition, seed = label.split("/")
    scores[(partition, seed)][method] = value

if not f1:
    print("Nothing to summarise yet.")
else:
    controls = ["random", "coordinate_grid", "pso", "ga"]
    for partition in sorted({p for p, _ in scores}):
        seeds = [s for (p, s) in scores if p == partition]
        print(f"=== {partition} ===")
        for method in ["aco", *controls]:
            vals = [scores[(partition, s)][method] for s in seeds if method in scores[(partition, s)]]
            if vals:
                m, s = mean_std(vals)
                print(f"  {method:16} n={len(vals)}  mean {m:.4f}  std {s:.4f}")
        for control in controls:
            pairs = [(scores[(partition, s)]["aco"], scores[(partition, s)][control])
                     for s in seeds
                     if "aco" in scores[(partition, s)] and control in scores[(partition, s)]]
            if pairs:
                wins = sum(a > c for a, c in pairs)
                diff = sum(a - c for a, c in pairs) / len(pairs)
                print(f"  aco vs {control:16} wins {wins}/{len(pairs)} seeds, mean diff {diff:+.4f}")
        print()

    print("Framing A stands unless make_tables (below) shows ACO SIGNIFICANTLY ahead of all four")
    print("controls. Losing to or tying any one of them is enough for framing A.")


In [ ]:
!python scripts/make_tables.py --results-dir results/fl/ablation --out paper/tables


---
# Before the session ends — SAVE, or you lose everything

Kaggle discards `/kaggle/working` unless the notebook is **committed**: use
**Save Version → Save & Run All (Commit)**, not the quick save. Next session, attach this
notebook's output as an input (**+ Add Input → Your Work**) so the restore cell brings it back
and every sweep resumes per-cell.


In [ ]:
from pathlib import Path

results = sorted(Path("results").rglob("*.json"))
print(f"{len(results)} result file(s) to save")
for directory in sorted({p.parent for p in results}):
    print(f"  {directory}: {len(list(directory.glob('*.json')))}")
print("\nSave Version -> Save & Run All (Commit). The quick save does NOT keep /kaggle/working.")
print("NEXT: Day 3 -- notebooks/kaggle_day3_main_and_robustness.ipynb, with Day 1's output attached.")
